In [13]:
import matplotlib.style as mplstyle
mplstyle.use('fast')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
from load_results import load, format
from scipy.stats import chi2 as chi2_dist
import itertools

from schedule_similarity import SCRIPTS, KS, NORMALIZE
from schedule_representations import experiments

import warnings

warnings.filterwarnings(
    "ignore",
    message=".*incompatible dtype.*", # for annotating significance
    category=FutureWarning
)

In [14]:
scores_to_plot = [
    "cka", 
    "spearman_rho",
    "rbo",
    "ndcg"]

In [15]:

df = load()
# # test for completeness of results
# SCRIPT_META = {
#     s: {
#         "module": s.replace(".py", "").replace("/", "."),
#         "label": s.split("/")[1].replace("run_", "").replace(".py", "")
#     }
#     for s in SCRIPTS
# }

# if_sorting_strategies = [
#     "most_harmful",
#     "most_helpful",
#     "least_influential",
#     "most_influential",
# ]


# df_pre = df.copy()

# group_keys = [
#     "Experiment",
#     "model",
#     "test_dataset",
#     "test_dataset_split",
#     "train_dataset",
#     "train_dataset_split",
#     "Rep_A",
#     "Rep_B",
#     "k",
#     "normalize",
# ]

# df_grouped = df_pre.groupby(group_keys, sort=False)


# for exp in experiments:
#     for script_a, script_b in itertools.combinations_with_replacement(SCRIPTS, 2):

#         meta_a = SCRIPT_META[script_a]
#         meta_b = SCRIPT_META[script_b]

#         for normalize in NORMALIZE:
#             for k in KS:

#                 base_key = (
#                     exp["name"],
#                     exp["model"],
#                     exp["dataset"],
#                     "test",
#                     exp["dataset"],
#                     "test",
#                     script_a.replace("/", ".").replace(".py", ""),
#                     script_b.replace("/", ".").replace(".py", ""),
#                     k,
#                     normalize,
#                 )
#                 if (script_a == script_b) and normalize:
#                     continue
#                 if (base_key not in df_grouped.groups):
#                     print(base_key)
#                     raise NotImplementedError

#                 df_subset_base = df_grouped.get_group(base_key)

#                 for if_sorting_strategy_a in if_sorting_strategies:
#                     for if_sorting_strategy_b in if_sorting_strategies:
#                         df_subset = df_subset_base[
#                             (
#                                 (df_subset_base["selection_strategy_name_a"] == if_sorting_strategy_a)
#                                 | (df_subset_base["selection_strategy_name_a"] == "-")
#                             )
#                             &
#                             (
#                                 (df_subset_base["selection_strategy_name_b"] == if_sorting_strategy_b)
#                                 | (df_subset_base["selection_strategy_name_b"] == "-")
#                             )
#                         ].dropna() # so that asserts fail on nan


#                         if normalize:
#                             if k == -1:
#                                 assert len(df_subset) == 2, display(df_subset)
#                             else:
#                                 assert len(df_subset) == 4, display(df_subset)
#                         else:
#                             if k == -1:
#                                 if script_a != script_b:
#                                     assert len(df_subset) == 6, display(df_subset)
#                                 else:
#                                     assert len(df_subset) == 4, display(df_subset)
#                             else:
#                                 assert len(df_subset) == 10, display(df_subset)

In [16]:
df = df[df["score_name"].isin(scores_to_plot)]
df = format(df)
df = df.drop(columns=["selection_strategy_name_a", "selection_strategy_name_b"])

In [17]:
assert len(df[df["score"].isna()]) == 0

In [18]:
long_df = df[["score_name", "base_model", "Experiment", "train_dataset", "Rep_A", "Rep_B", "test_type", "k", "normalize", "score"]].copy().rename(columns={"score_name": "metric", "train_dataset": "dataset"})
long_df["Rep_pair"] = long_df.apply(
    lambda r: ":".join(sorted([str(r["Rep_A"]), str(r["Rep_B"])])),
    axis=1
)
long_df = long_df.drop(columns=["Rep_A", "Rep_B"])
assert len(long_df[long_df.isna().any(axis=1)]) == 0
long_df

,metric,base_model,Experiment,dataset,test_type,k,normalize,score,Rep_pair
0,cka,Qwen3-8B-Base,ftrace,loris3/ftrace_date-lm_conversational-sample,rep,-1,False,0.729123,$r^{L2}_{\nabla f}$:$r_{\nabla f}^{aa}$
5,cka,Qwen3-8B-Base,ftrace,loris3/ftrace_date-lm_conversational-sample,rep,-1,False,0.017250,$r^{L2}_{\nabla f}$:$r_{bm25}$
8,cka,Qwen3-8B-Base,ftrace,loris3/ftrace_date-lm_conversational-sample,rep,-1,False,0.104842,$r^{L2}_{\nabla f}$:qwen
12,cka,Qwen3-8B-Base,ftrace,loris3/ftrace_date-lm_conversational-sample,rep,-1,False,0.179203,$r^{L2}_{\nabla f}$:$r_{tfidf}$
17,cka,Qwen3-8B-Base,ftrace,loris3/ftrace_date-lm_conversational-sample,rep,-1,False,0.157773,$r^{L2}_{\nabla f}$:nemot
...,...,...,...,...,...,...,...,...,...
54611,spearman_rho,Qwen3-8B-Base,squad_v2,loris3/squad_v2_conversational-sample,func,-1,False,-1.000000,$r_{\nabla f}^{aa} {harm}$:$r_{\nabla f}^{aa} ...
54612,ndcg,Qwen3-8B-Base,squad_v2,loris3/squad_v2_conversational-sample,func,-1,False,0.824915,$r_{\nabla f}^{aa} {harm}$:$r_{\nabla f}^{aa} ...
54614,rbo,Qwen3-8B-Base,squad_v2,loris3/squad_v2_conversational-sample,func,-1,False,0.307353,$r_{\nabla f}^{aa} {harm}$:$r_{\nabla f}^{aa} ...
54615,spearman_rho,Qwen3-8B-Base,squad_v2,loris3/squad_v2_conversational-sample,func,-1,False,1.000000,$r^{L2}_{\nabla f} {help}$:$r^{L2}_{\nabla f} ...


$y_{i,m,d}=\beta_0+\beta^{(m)}_{m}+\beta^{(d)}_{d}+\beta^{(m \times d)}_{m,d}+u_i+\epsilon_{i,m,d}$

In [19]:
def fit_model(definition, long_df, score_name, k, normalize ):
    if normalize:
        return None
    df = long_df[(long_df["metric"] == score_name) & (long_df["k"] == k)  & (long_df["normalize"] == normalize)].copy()
    
    cols = ["base_model", "dataset", "score", "Rep_pair"]
    nan_rows = df[df[cols].isnull().any(axis=1)]
    
    if df[cols].isnull().values.any():
        print(f"missing entries for {score_name} at k={k}")
        display(nan_rows)
        print(len(df),len(nan_rows))
        return None
    if (df[cols].nunique() == 1).any():
        print(f"constant entries for {score_name} at k={k}")
        return None
    data = df[cols].reset_index(drop=True)
    # display(data)
    # print(data.groupby("Rep_pair").size().describe())
    return smf.mixedlm(
                definition,
                data=data,
                groups=data["Rep_pair"]
            ).fit(reml=False)

configs = long_df.groupby(["metric", "k", "test_type", "normalize"]).groups.keys()
models = {
    (score_name, k, test_type, normalize) : fit_model("score ~ C(base_model) * C(dataset)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize in configs
} 

models_no_interaction = {
    (score_name, k, test_type, normalize) : fit_model("score ~ C(base_model) + C(dataset)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize  in configs
} 

models_no_base_model = {
    (score_name, k, test_type, normalize) : fit_model("score ~ C(dataset)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize  in configs
} 

models_no_dataset = {
    (score_name, k, test_type, normalize) : fit_model("score ~ C(base_model)", long_df, score_name, k, normalize) for score_name, k, test_type, normalize  in configs
} 


$H_0: \beta^{(m \times d)}_{m,d} = 0 \quad \forall m,d,$



$H_0^{(M)}: \beta^{(m)}_{m} = 0 \quad \forall m$

$H_0^{(D)}: \beta^{(d)}_{d} = 0 \quad \forall d$

In [20]:


def lr_test_df(full_model, reduced_model, term_name):
    lr_stat = 2 * (full_model.llf - reduced_model.llf)

    df_diff = full_model.df_modelwc - reduced_model.df_modelwc
    p_value = chi2_dist.sf(lr_stat, df_diff)

    return pd.DataFrame({
        term_name: [term_name],
        "chi2": [lr_stat],
        "P>chi2": [p_value],
        "df constraint": [df_diff]
    }).set_index(term_name)
def compute_r2_mixedlm(model):
    # fixed effects design matrix
    fe = model.model.exog
    fe_params = model.fe_params.values

    # fixed-effect linear predictor η
    eta = fe @ fe_params
    sigma_f2 = np.var(eta, ddof=1)

    # random intercept variance (Rep_pair)
    sigma_pair2 = model.cov_re.values.item()

    # residual variance
    sigma_e2 = model.scale

    denom = sigma_f2 + sigma_pair2 + sigma_e2

    R2_m = sigma_f2 / denom
    R2_c = (sigma_f2 + sigma_pair2) / denom

    return sigma_f2, sigma_pair2, sigma_e2, R2_m, R2_c


result_dfs_lrt = []
result_dfs_r2 = []
for (score_name, k, test_type, normalize), model in models.items():
    if model is None:
        continue
    interaction_test = pd.concat([
        lr_test_df(
            model,
            models_no_base_model[(score_name, k, test_type, normalize)],
            "C(base_model)"
        ),
        lr_test_df(
            model,
            models_no_dataset[(score_name, k, test_type,normalize)],
            "C(dataset)"
        ),
        lr_test_df(
            model,
            models_no_interaction[(score_name, k, test_type,normalize)],
            "C(base_model):C(dataset)"
        )
        ])
    # display(interaction_test)
    df_model_dataset = interaction_test.stack().rename_axis(["term", "stat"]).to_frame().T.reset_index(drop=True)
    df_model_dataset["metric"] = score_name
    df_model_dataset["test_type"] = test_type
    df_model_dataset["k"] = k
    df_model_dataset["normalize"] = normalize
    df_model_dataset = df_model_dataset.set_index(["metric", "k", "normalize"])
    result_dfs_lrt.append(df_model_dataset)
    
    r2_dfs = []
    for setting, model_ in [
        ("full", model),
        ("no_interaction", models_no_interaction[(score_name, k,test_type, normalize)]),
        ("no_base_model", models_no_base_model[(score_name, k,test_type,normalize)]),
        ("no_dataset", models_no_dataset[(score_name, k,test_type,normalize)])
        ]:
        sigma_f2, sigma_pair2, sigma_e2, R2_m, R2_c = compute_r2_mixedlm(model_)
        df_r2 = pd.DataFrame({

            "metric": [score_name],
            "test_type": [test_type],
            "normalize": [normalize],
            "k": [k],
            # "sigma_f2": [sigma_f2],
            # "sigma_pair2": [sigma_pair2],
            # "sigma_e2": [sigma_e2],
            "R2_m": [R2_m],
            "R2_c": [R2_c]
        })
        df_r2.columns = pd.MultiIndex.from_product(
            [[setting], df_r2.columns]
        )

        r2_dfs.append(df_r2)
    result_dfs_r2.append(pd.concat(r2_dfs, axis=1))
        
    

    
 
df_lrt = pd.concat(result_dfs_lrt)
df_r2_model_dataset = pd.concat(result_dfs_r2)




# LRT results

In [21]:
df_display = df_lrt.reset_index().set_index(
    ["test_type", "metric", "k"]
).sort_values(["test_type", "metric", "k"])

df_display = df_display.loc[
    :, df_display.columns.get_level_values("stat") != "df constraint"
]
df_display.index = df_display.index.set_names(
    ["Test type", "Metric \\rotatebox[origin=c]{270}{$\Rsh$}", "k"]
)
df_display = df_display.rename(columns={
    "C(base_model)": "Base model",
    "C(dataset)": "Dataset",
    "C(base_model):C(dataset)": "Interaction"
})
df_display.columns = df_display.columns.set_names(["Term $\\rightarrow$", "stat"])


chi2_df = df_display.xs("chi2", level="stat", axis=1)
p_df = df_display.xs("P>chi2", level="stat", axis=1)


def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

stars_df = p_df.map(p_to_star)


display_df = df_display.copy()

mask_p = display_df.columns.get_level_values("stat") == "P>chi2"


display_df.loc[:, mask_p] = stars_df.values


mask_chi2 = display_df.columns.get_level_values("stat") == "chi2"


styled = (
    display_df.style
    .background_gradient(
        cmap="viridis",
        subset=display_df.columns[mask_chi2]
    )
    .hide(axis="columns", level="stat")
    .hide(axis="columns", subset=[ "normalize"])
    .hide(axis="index", level=[0,2])
).format_index(escape="latex", axis=1).format_index(escape="latex", axis=0)\
    .format(precision=1) 
display(styled)
latex = styled.to_latex(convert_css=True, hrules=True, multicol_align="l",
                                   column_format='l' + 'l' * len(styled.data.columns))
with open("figures/lrt-table.tex", "w", encoding="utf-8") as f:
            f.write(latex)
            


In [22]:
df = df_r2_model_dataset.copy()

df = df.set_index(("full", "metric"))

delta_r2 = pd.DataFrame(index=df.index)

delta_r2["interaction_model_dataset_R2_m"] = (
    df[("full", "R2_m")] - df[("no_interaction", "R2_m")]
)
delta_r2["interaction_model_dataset_R2_c"] = (
    df[("full", "R2_c")] - df[("no_interaction", "R2_c")]
)

delta_r2["base_model_R2_m"] = (
    df[("full", "R2_m")] - df[("no_base_model", "R2_m")]
)
delta_r2["base_model_R2_c"] = (
    df[("full", "R2_c")] - df[("no_base_model", "R2_c")]
)

delta_r2["dataset_R2_m"] = (
    df[("full", "R2_m")] - df[("no_dataset", "R2_m")]
)
delta_r2["dataset_R2_c"] = (
    df[("full", "R2_c")] - df[("no_dataset", "R2_c")]
)



# r2_m_cols = [col for col in delta_r2.columns if "R2_m" in col]


# delta_r2.style.background_gradient(
#     cmap="viridis",
#     subset=r2_m_cols,
#     axis=1
# )

## $\Delta R^2$

In [23]:
import pandas as pd

delta_r2_table = pd.DataFrame({
    "Base model": delta_r2["base_model_R2_m"],
    "Dataset": delta_r2["dataset_R2_m"],
    "Interaction": delta_r2["interaction_model_dataset_R2_m"],
})
delta_r2_table.index.name = None
delta_r2_table.index = df_display.index.set_names(
    ["Test type", "Metric \\rotatebox[origin=c]{270}{$\Rsh$}", "k"]
)
delta_r2_table.columns = delta_r2_table.columns.set_names(["Term $\\rightarrow$"])
styled = (
    delta_r2_table.style
    .background_gradient(
        cmap="viridis",
        axis=1  
    )
    .format("{:.3f}").format_index(escape="latex", axis=1).format_index(escape="latex", axis=0)
).hide(axis="index", level=[0,2])
display(styled)

latex = styled.to_latex(convert_css=True, hrules=True, multicol_align="l",
                                   column_format='l' + 'l' * len(styled.data.columns))
with open("figures/delta-r2-table.tex", "w", encoding="utf-8") as f:
            f.write(latex)


Term $\rightarrow$,Base model,Dataset,Interaction
Metric \rotatebox[origin=c]{270}{$\Rsh$},,,
ndcg,0.010,0.007,0.002
rbo,0.002,0.017,0.001
spearman\_rho,0.000,0.000,0.000
cka,0.000,0.000,0.000


In [24]:
# df = df_r2_model_dataset.copy()

# df = df.set_index(("full", "metric"))

# r2_table = pd.DataFrame(index=df.index)

# r2_table["$R^2_m$"] = df[("full", "R2_m")]
# r2_table["$R^2_c$"] = df[("full", "R2_c")]

# r2_m_cols = [col for col in r2_table.columns if "$R^2_m$" in col]
# r2_table.index.name = None
# r2_table.style.background_gradient(
#     cmap="viridis",
#     axis=1
# ).format_index(escape="latex", axis=1).format_index(escape="latex", axis=0)
# latex = styled.to_latex(convert_css=True, hrules=True, multicol_align="l",
#                                    column_format='l' + 'l' * len(styled.data.columns))
# display(styled)
# with open("figures/delta-r2-table.tex", "w", encoding="utf-8") as f:
#             f.write(latex)

### 